# Giga Meter — Drop-off Analysis

Who stopped measuring, when, and what distinguishes them. Companion to
`meter_explorer_02.ipynb`; loads the clean dataset from
`download_data_01.ipynb` when present, and falls back to the raw
country parquet otherwise.

**Questions**
- **Q1** — how many schools and devices have sent nothing in the last 30 days / 6 months / year, and how many never measured at all?
- **Q2** — when did they stop: drop-off timing, tenure before dropping, and cohort retention by install month.
- **Q3** — what kind of school drops: education level, region, ISP, device count, app version, tenure.
- **Q4** — why: did performance degrade before the last measurement, and do the correlates hold up statistically?

**Definitions.** Recency is measured against the **last date in the data**, not today
(the consolidated table lags ~1 day). A *school* is silent if none of its devices
reported; a *device* is a distinct `browser_id` (Giga Meter installs one per browser
profile / machine — `device_id` is sparsely populated). Activity uses the
**unfiltered** measurement frame: a school that only reached a secondary M-Lab server
is still active, even though the analysis frame excludes those rows.

---
## Part 0 — Setup & data

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================

import os
import sys
import json
from pathlib import Path
from datetime import date, timedelta

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import pytz

from IPython.display import display

# Optional connectivity libs (only needed when USE_CACHED_DATA = False)
try:
    import delta_sharing
    DELTA_SHARING_AVAILABLE = True
except ImportError:
    DELTA_SHARING_AVAILABLE = False
    print("⚠️ delta_sharing not available - will use cached data only")

try:
    import trino
    from trino.dbapi import connect
    TRINO_AVAILABLE = True
except ImportError:
    TRINO_AVAILABLE = False
    print("⚠️ trino not available - will use cached data only")

# -----------------------------------------------------------------------------
# Data-loading helpers (bundled in ./helpers)
#   load_master            - school master via Delta Sharing / Trino, CSV-cached
#   load_measurements      - country measurements via Trino, parquet-cached + incremental
#   format_measurements    - query builder + light post-processing for the
#                            consolidated table default.all_gigameter_measurement_data
#   get_trino_cursor/engine - PRD Trino over kubectl port-forward (auto-started)
# -----------------------------------------------------------------------------
set_up_dir = Path.cwd() / "helpers"
if str(set_up_dir) not in sys.path:
    sys.path.insert(0, str(set_up_dir))

try:
    import format_measurements
    from load_master import load_master, load_master_trino
    from load_measurements import (
        load_measurements,
        load_registration,
        get_trino_cursor,
        get_trino_engine,
    )
    HELPERS_AVAILABLE = True
except ImportError as e:
    HELPERS_AVAILABLE = False
    print(f"⚠️ data-loading helpers not importable from {set_up_dir}: {e}")

# -----------------------------------------------------------------------------
# Analysis helpers + Giga chart style (moved out of the notebook)
#   eda_helpers      - education inference, ISP canonicalisation, IQB-Edu engine,
#                      legacy service-tier scaffolding
#   giga_chart_style - fonts + palette + rcParams (applied on import)
# -----------------------------------------------------------------------------
from eda_helpers import (resolve_country, infer_edlevel_from_name, clean_isp, build_isp_canon,
                         IQB_CONFIG, IQB_USE_CASES, IQB_PERCENTILES, IQB_BENCHMARK,
                         MIN_MEASUREMENTS_FOR_IQB, calculate_iqb_score,
                         _config_for_use_case,
                         classify_service_level, tier_order,
                         TIER_THRESHOLD_1, TIER_THRESHOLD_2, TIER_THRESHOLD_3,
                         paired_shift_test, two_group_shift_test,
                         format_shift_result, bootstrap_ci,
                         wilson_ci, fmt_pct_ci,
                         kruskal_omnibus, pairwise_shift_tests)
from giga_chart_style import (GIGA_PRIMARY, GIGA_GREY, GIGA_BLUE, GIGA_GOOD,
                              GIGA_MODERATE, GIGA_BAD, GIGA_TIER_RAMP, GIGA_CYCLE,
                              GIGA_SUPTITLE)

print("\u2713 Imports complete \u00b7 Giga chart style applied (Open Sans / Manrope, Giga palette)")


In [ ]:
# =============================================================================
# LOAD — clean dataset if available, else the raw country parquet
# =============================================================================
import json as _json
from pathlib import Path

COUNTRY_NAME = "Fiji"          # <- set the country
CACHE_DIR = f"./cache/{COUNTRY_NAME}"   # cleaned data + caches land here (gitignored)
_slug = COUNTRY_NAME.lower().replace(' ', '')
_cd = Path(CACHE_DIR)

_params_p = _cd / f"{_slug}_clean_params.json"
if _params_p.exists():
    PARAMS = _json.loads(_params_p.read_text())
    COUNTRY_ISO3, COUNTRY_ISO2 = PARAMS['country']['iso3'], PARAMS['country']['iso2']
    TIMEZONE = PARAMS['country']['timezone']
    # activity questions want EVERY measurement, so prefer the unfiltered frame
    m_all = pd.read_parquet(_cd / f"{_slug}_clean_unfiltered.parquet")
    m = pd.read_parquet(_cd / f"{_slug}_clean.parquet")          # filtered: performance metrics
    _src_note = f"clean dataset prepared {PARAMS['generated_at']}"
else:
    from eda_helpers import resolve_country
    _c = resolve_country(COUNTRY_NAME)
    COUNTRY_ISO3, COUNTRY_ISO2, TIMEZONE = _c['iso3'], _c['iso2'], _c['timezone']
    m_all = pd.read_parquet(_cd / f"{_slug}_measurements.parquet")
    for _c2 in ('download_speed', 'upload_speed', 'latency', 'packet_loss_rate'):
        if _c2 in m_all.columns:
            m_all[_c2] = pd.to_numeric(m_all[_c2], errors='coerce')
            m_all.loc[m_all[_c2] < 0, _c2] = np.nan
    if 'latency' in m_all.columns:
        m_all.loc[m_all['latency'] >= 4_294_967, 'latency'] = np.nan
    m_all['loss_rate'] = m_all.get('packet_loss_rate')
    m = m_all
    _src_note = "raw country parquet (no clean dataset yet — run download_data_01.ipynb)"

# `date` as shipped is the UTC calendar day (upstream does DATE_TRUNC on created_at
# with no timezone applied), which misdates 66% of Fiji rows and inflates school-day
# counts by 7.6%. Rebuild it from created_timestamp + the country's zone; the raw
# value survives as `date_utc`. Every day/gap/cadence metric below is local.
from format_measurements import localise_dates
_same_frame = m is m_all
m_all = localise_dates(m_all, timezone=TIMEZONE)
m = m_all if _same_frame else localise_dates(m, timezone=TIMEZONE, verbose=False)
for _df in (m_all, m):
    _df['date'] = pd.to_datetime(_df['date'])
master = pd.read_csv(_cd / f"{COUNTRY_ISO3}_master_datapull.csv", low_memory=False)
r = pd.read_parquet(_cd / f"{_slug}_registered.parquet")

REF_DATE = m_all['date'].max()          # recency anchored to the data, not today
print(f"✓ {COUNTRY_NAME} ({COUNTRY_ISO3}) — {_src_note}")
print(f"  {len(m_all):,} measurements from {m_all['school_id_giga'].nunique():,} schools, "
      f"{m_all['date'].min():%d %b %Y} → {REF_DATE:%d %b %Y} (reference date)")
print(f"  registration rows: {len(r):,} | master schools: {len(master):,}")

---
## Q1 — How many are silent?

Two units, because they answer different questions: **schools** (does the Ministry
still get data from this school?) and **devices** (how many installs went quiet —
a school can lose one of three devices and keep reporting).

In [ ]:
# =============================================================================
# Q1 — SILENCE BY RECENCY, SCHOOLS AND DEVICES
# =============================================================================
_DEV = 'browser_id' if m_all['browser_id'].notna().any() else 'device_id'

sch_last = m_all.groupby('school_id_giga')['date'].agg(first='min', last='max', n='size')
sch_last['days_silent'] = (REF_DATE - sch_last['last']).dt.days
dev_last = (m_all.dropna(subset=[_DEV]).groupby(_DEV)
            .agg(school_id_giga=('school_id_giga', 'first'), first=('date', 'min'),
                 last=('date', 'max'), n=('date', 'size')))
dev_last['days_silent'] = (REF_DATE - dev_last['last']).dt.days

_mapped = master['school_id_giga'].nunique()
_tiers = [('active (≤30d)', 0, 30), ('at risk (31–90d)', 31, 90),
          ('dormant (91–365d)', 91, 365), ('lost (>365d)', 366, 10**6)]
rows = []
for _lbl, _lo, _hi in _tiers:
    _s = ((sch_last['days_silent'] >= _lo) & (sch_last['days_silent'] <= _hi)).sum()
    _d = ((dev_last['days_silent'] >= _lo) & (dev_last['days_silent'] <= _hi)).sum()
    rows.append({'status': _lbl, 'schools': _s, '% of measuring schools': round(100*_s/len(sch_last), 1),
                 'devices': _d, '% of devices': round(100*_d/len(dev_last), 1)})
rows.append({'status': 'never measured', 'schools': _mapped - len(sch_last),
             '% of measuring schools': np.nan, 'devices': np.nan, '% of devices': np.nan})
silence = pd.DataFrame(rows).set_index('status')
print(f"Reference date {REF_DATE:%d %b %Y} · {len(sch_last):,} schools and {len(dev_last):,} devices "
      f"({_DEV}) have measured at least once; {_mapped:,} schools are mapped in the master.")
display(silence)

_q = lambda days: (int((sch_last['days_silent'] > days).sum()), int((dev_last['days_silent'] > days).sum()))
for _d in (30, 182, 365):
    _s, _dv = _q(_d)
    print(f"  Nothing sent in the last {_d:>3} days: {_s:>4} schools ({100*_s/len(sch_last):>4.1f}%) · "
          f"{_dv:>4} devices ({100*_dv/len(dev_last):>4.1f}%)")

# Device attrition INSIDE schools that are still reporting — a school can lose
# installs and stay "active" on the survivors, so school-level status hides it.
_ds = dev_last.groupby('school_id_giga')['days_silent'].agg(
    devices='size', silent=lambda s: int((s > 30).sum()))
_ds['school_active'] = ~_ds.index.map(sch_last['days_silent'] > 30)
_act_sch = _ds[_ds['school_active']]
_hit = _act_sch[_act_sch['silent'] > 0]

print(f"\nDEVICE ATTRITION INSIDE STILL-ACTIVE SCHOOLS")
print(f"  {len(_act_sch):,} schools are still reporting (school measured in the last 30 days).")
print(f"  Of those, {len(_hit):,} ({100*len(_hit)/max(len(_act_sch),1):.0f}%) have at least one "
      f"install that has gone quiet.")
print(f"  Those {len(_hit):,} schools hold {int(_hit['devices'].sum()):,} installs, of which "
      f"{int(_hit['silent'].sum()):,} are silent >30d — they keep reporting on the remaining "
      f"{int((_hit['devices'] - _hit['silent']).sum()):,}.")
print(f"  Typical affected school: {_hit['devices'].median():.0f} installs, "
      f"{_hit['silent'].median():.0f} of them quiet.")
_solo = int((( _act_sch['devices'] - _act_sch['silent']) == 1).sum())
print(f"  {_solo:,} active schools ({100*_solo/max(len(_act_sch),1):.0f}%) are down to a SINGLE working "
      f"install — one machine away from going silent.")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, _d, _lbl in ((axes[0], sch_last, 'schools'), (axes[1], dev_last, 'devices')):
    _counts = [((_d['days_silent'] >= lo) & (_d['days_silent'] <= hi)).sum() for _, lo, hi in _tiers]
    ax.bar([t[0] for t in _tiers], _counts,
           color=[GIGA_GOOD, GIGA_MODERATE, GIGA_PRIMARY[300], GIGA_BAD], edgecolor='white')
    for _x, _v in enumerate(_counts):
        ax.text(_x, _v, f'{_v:,}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_ylabel(_lbl); ax.set_title(f'{_lbl.title()} by silence tier — {COUNTRY_NAME}')
    ax.tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.show()

---
## Q2 — When did they stop?

Three views: the calendar month schools went quiet, how long they lasted before
going quiet (tenure), and cohort retention — of the schools that started measuring
in month *X*, what share were still measuring *n* months later.

---
## Calibrating "dropped"

Schools pause and resume, so a fixed 30-day cut-off would label many
temporary pauses as churn. The threshold is picked from how often silence
of a given length actually ended in a return.


In [ ]:
# =============================================================================
# DROP-OFF THRESHOLD — CALIBRATED WITH SURVIVAL ANALYSIS (exposure-correct)
# =============================================================================
# "Silent for N days" only means "dropped" if silence of that length rarely ends
# in a return. Counting events would be biased: a school installed last month
# CANNOT contribute a 180-day pause, so long bands have less exposure than short
# ones. Kaplan-Meier fixes this — every silence spell either ended (the school
# returned) or is still running at the reference date (censored), and censored
# spells stay in the risk set until they drop out instead of counting as
# failures. The decision-relevant output is CONDITIONAL: given a school has
# already been quiet X days, how likely is it to come back?
MAX_RESUME_RATE = 25          # a threshold is "decisive" when fewer than this % return

_sdd = m_all.drop_duplicates(['school_id_giga', 'date']).sort_values(['school_id_giga', 'date'])
_spells = []
for _s, _g in _sdd.groupby('school_id_giga'):
    _d = _g['date'].tolist()
    for _i in range(1, len(_d)):
        if (_d[_i] - _d[_i - 1]).days >= 2:
            _spells.append(((_d[_i] - _d[_i - 1]).days, 1))          # completed: returned
    if (REF_DATE - _d[-1]).days >= 2:
        _spells.append(((REF_DATE - _d[-1]).days, 0))                # ongoing: censored
_sp = pd.DataFrame(_spells, columns=['dur', 'returned'])

# Kaplan-Meier survivor S(t) = P(still silent at day t)
_S, _km = 1.0, {}
for _t in np.sort(_sp.loc[_sp['returned'] == 1, 'dur'].unique()):
    _n = int((_sp['dur'] >= _t).sum())
    _dd = int(((_sp['dur'] == _t) & (_sp['returned'] == 1)).sum())
    _S *= (1 - _dd / _n); _km[_t] = _S
_KM = pd.Series(_km)
def _surv(x):
    _s = _KM[_KM.index <= x]
    return float(_s.iloc[-1]) if len(_s) else 1.0

# Short thresholds matter too: a school quiet for 7 or 14 days may already be
# recoverable with a nudge, long before it looks like churn.
_cand = [7, 14, 21, 30, 45, 60, 90, 120, 182, 270, 365]
_cal = pd.DataFrame({'silent_for_days': _cand})
_cal['risk set (spells)'] = [int((_sp['dur'] >= X).sum()) for X in _cand]
for _h in (7, 14, 21):
    _cal[f'returns within +{_h}d %'] = [round(100 * (1 - _surv(X + _h) / _surv(X))) for X in _cand]
_cal['returns within +30d %'] = [round(100 * (1 - _surv(X + 30) / _surv(X))) for X in _cand]
_cal['returns within +90d %'] = [round(100 * (1 - _surv(X + 90) / _surv(X))) for X in _cand]
_cal['returns by day 365 %'] = [round(100 * (1 - _surv(365) / _surv(X))) for X in _cand]
_cal['schools now at this point'] = [int((sch_last['days_silent'] >= X).sum()) for X in _cand]

_ok = _cal[(_cal['returns by day 365 %'] <= MAX_RESUME_RATE) & (_cal['silent_for_days'] >= 30)]
DROP_DAYS = int(_ok['silent_for_days'].iloc[0]) if len(_ok) else 182   # <- OVERRIDE here if needed

print(f"{len(_sp):,} silence spells ≥2d — {int(_sp['returned'].sum()):,} completed, "
      f"{int((1 - _sp['returned']).sum()):,} still running (censored at {REF_DATE:%d %b %Y})")
print(f"Typical rhythm: {100 * (_sp['dur'] <= 7).mean():.0f}% of all pauses are ≤7 days (weekends and "
      f"the like), so unconditional return rates say little — read the conditional table:\n")
display(_cal.set_index('silent_for_days'))

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(_cal['silent_for_days'], _cal['returns by day 365 %'], marker='o', color=GIGA_PRIMARY[600],
        label='returns by day 365')
ax.plot(_cal['silent_for_days'], _cal['returns within +90d %'], marker='s', ls='--',
        color=GIGA_PRIMARY[300], label='returns within +90d')
ax.axhline(MAX_RESUME_RATE, color=GIGA_BAD, ls='--', lw=1, label=f'{MAX_RESUME_RATE}% decision line')
ax.axvline(DROP_DAYS, color=GIGA_GREY[800], ls=':', lw=1.5, label=f'chosen: {DROP_DAYS}d')
ax.set_xlabel('days already silent'); ax.set_ylabel('% that return (Kaplan-Meier)')
ax.set_title(f'Does silence end in a return? — {COUNTRY_NAME}')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

# Churn INCIDENCE — rate per unit of exposure, comparable across countries/periods
_obs_years = float((REF_DATE - sch_last['first']).dt.days.sum()) / 365.25
_churn_events = int(((_sp['dur'] >= 182) & (_sp['returned'] == 0)).sum())
CHURN_PER_100_SCHOOL_YEARS = 100 * _churn_events / _obs_years
print(f"→ DROP_DAYS = {DROP_DAYS}: only "
      f"{int(_cal.loc[_cal['silent_for_days'] == DROP_DAYS, 'returns by day 365 %'].iloc[0])}% of schools "
      f"silent this long return within the year "
      f"({int(_cal.loc[_cal['silent_for_days'] == DROP_DAYS, 'schools now at this point'].iloc[0]):,} schools "
      f"are at or past it today).")
print(f"   A 30-day rule would flag "
      f"{int(_cal.loc[_cal['silent_for_days'] == 30, 'schools now at this point'].iloc[0]):,} schools, but "
      f"{int(_cal.loc[_cal['silent_for_days'] == 30, 'returns by day 365 %'].iloc[0])}% of them return — "
      f"too many false alarms to drive outreach.")
print(f"   Churn incidence: {CHURN_PER_100_SCHOOL_YEARS:.1f} events per 100 school-years observed "
      f"({_churn_events:,} spells silent >182d over {_obs_years:,.0f} school-years) — use this, not a "
      f"share of pauses, to compare countries or periods.")
print(f"   Follow-up limit: the data spans {(REF_DATE - m_all['date'].min()).days / 365.25:.1f} years, so "
      f"returns beyond that horizon are unobservable and long-silence return rates are lower bounds.")

---
## Season-aware drop-off classification

The break period is learned from each country own rhythm, so the same
logic works north or south of the equator.


In [ ]:
# =============================================================================
# SEASON-AWARE CLASSIFICATION — holiday months learned from the data
# =============================================================================
# School calendars differ by hemisphere and country, so the low season is
# DERIVED, not assumed: for each calendar month, average measuring days per
# school-month IN TENURE (exposure-corrected — raw month counts are biased when
# the data covers unequal numbers of each calendar month). Months below
# LOW_SEASON_FRAC of the peak are that country's break period.
LOW_SEASON_FRAC = 0.70     # month counts as "low season" below this share of peak
SEASON_GRACE_DAYS = 120    # a silence spanning the break, up to this long, is a pause
EARLY_LIFE_DAYS = 30       # stopped within this many days of first measuring

_exp = pd.DataFrame([(s, p) for s, (a, b) in sch_last[['first', 'last']].iterrows()
                     for p in pd.period_range(a.to_period('M'), b.to_period('M'), freq='M')],
                    columns=['school_id_giga', 'mo'])
_actm = (m_all.assign(mo=m_all['date'].dt.to_period('M'))
         .drop_duplicates(['school_id_giga', 'date'])
         .groupby(['school_id_giga', 'mo']).size().rename('days').reset_index())
_j = _exp.merge(_actm, on=['school_id_giga', 'mo'], how='left').fillna({'days': 0})
_season = _j.assign(cm=_j['mo'].dt.month).groupby('cm')['days'].mean()
LOW_SEASON_MONTHS = sorted(_season[_season < LOW_SEASON_FRAC * _season.max()].index)
_MON = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

fig, ax = plt.subplots(figsize=(10, 3.2))
ax.bar(_season.index, _season.values,
       color=[GIGA_MODERATE if c in LOW_SEASON_MONTHS else GIGA_PRIMARY[600] for c in _season.index],
       edgecolor='white')
ax.axhline(LOW_SEASON_FRAC * _season.max(), color=GIGA_GREY[700], ls='--', lw=1,
           label=f'{LOW_SEASON_FRAC:.0%} of peak')
ax.set_xticks(range(1, 13)); ax.set_xticklabels(_MON, fontsize=8)
ax.set_ylabel('measuring days per school-month'); ax.legend(fontsize=8)
ax.set_title(f"Seasonal rhythm (exposure-corrected) — {COUNTRY_NAME}: low season = "
             + ', '.join(_MON[c - 1] for c in LOW_SEASON_MONTHS))
plt.tight_layout(); plt.show()

def _spans_low_season(last_date, days_silent):
    """True when most months covered by the silence fall in the low season."""
    _months = pd.period_range(last_date.to_period('M'),
                              (last_date + pd.Timedelta(days=days_silent)).to_period('M'), freq='M')
    _in = [p.month in LOW_SEASON_MONTHS for p in _months]
    return bool(_in) and (sum(_in) / len(_in) >= 0.5)

_cls = sch_last.copy()
_cls['tenure_days'] = (_cls['last'] - _cls['first']).dt.days
_cls['seasonal'] = [_spans_low_season(l, d) for l, d in zip(_cls['last'], _cls['days_silent'])]

def _classify(r):
    if r['days_silent'] <= 30:                                   return 'active'
    if r['tenure_days'] <= EARLY_LIFE_DAYS:                      return 'early-life failure'
    if r['seasonal'] and r['days_silent'] <= SEASON_GRACE_DAYS:  return 'seasonal pause'
    if r['days_silent'] <= 90:                                   return 'at risk (31-90d)'
    if r['days_silent'] <= 182:                                  return 'dormant (91-182d)'
    return 'lost (>182d)'

_cls['dropoff_class'] = _cls.apply(_classify, axis=1)
CLASS = _cls['dropoff_class']
DROPPED_CLASSES = ['dormant (91-182d)', 'lost (>182d)', 'early-life failure']
_ORDER = ['active', 'seasonal pause', 'at risk (31-90d)', 'dormant (91-182d)',
          'lost (>182d)', 'early-life failure']

_tbl = CLASS.value_counts().reindex(_ORDER).fillna(0).astype(int).rename('schools').to_frame()
_tbl['% of measuring schools'] = (100 * _tbl['schools'] / len(_cls)).round(1)
print(f"Low season learned from the data: {', '.join(_MON[c - 1] for c in LOW_SEASON_MONTHS)} "
      f"(trough {_season.min():.1f} vs peak {_season.max():.1f} measuring days per school-month)")
display(_tbl)
_naive = int((sch_last['days_silent'] > 30).sum())
_real = int(CLASS.isin(DROPPED_CLASSES).sum())
print(f"A flat >30-day rule would call {_naive:,} schools dropped; the season-aware classes put "
      f"{_real:,} in a genuinely dropped state.")
print("The difference is seasonal pauses plus the 31-90d at-risk band, where most silences "
      "historically end in a return.")

In [ ]:
# =============================================================================
# Q2 — DROP-OFF TIMING, TENURE, COHORT RETENTION
# =============================================================================
# DROP_DAYS comes from the calibration cell above.
dropped = sch_last[sch_last['days_silent'] > DROP_DAYS].copy()
dropped['last_month'] = dropped['last'].dt.to_period('M').dt.to_timestamp()
dropped['tenure_days'] = (dropped['last'] - dropped['first']).dt.days

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
_by_month = dropped.groupby('last_month').size()
axes[0].bar(_by_month.index, _by_month.values, width=20, color=GIGA_BAD, alpha=0.85)
axes[0].set_ylabel('schools whose LAST measurement fell in that month')
axes[0].set_title(f'When schools went quiet — {COUNTRY_NAME}')
axes[0].tick_params(axis='x', rotation=60)
axes[1].hist(dropped['tenure_days'] / 30.44, bins=30, color=GIGA_PRIMARY[600], edgecolor='white')
axes[1].axvline(dropped['tenure_days'].median() / 30.44, color=GIGA_GREY[800], ls='--',
                label=f"median {dropped['tenure_days'].median()/30.44:.1f} months")
axes[1].set_xlabel('months between first and last measurement')
axes[1].set_ylabel('schools'); axes[1].set_title('Tenure before going quiet'); axes[1].legend()
plt.tight_layout(); plt.show()

_short = (dropped['tenure_days'] <= 30).sum()
print(f"{len(dropped):,} schools silent >{DROP_DAYS}d. Tenure before going quiet: "
      f"median {dropped['tenure_days'].median()/30.44:.1f} months, "
      f"{_short:,} ({100*_short/len(dropped):.0f}%) lasted a month or less.")
print("Peak drop-off months:")
display(_by_month.sort_values(ascending=False).head(5).rename('schools').to_frame())

# Cohort retention: share of each install-month cohort still measuring n months on
_act = m_all[['school_id_giga', 'date']].copy()
_act['month'] = _act['date'].dt.to_period('M')
_act = _act.drop_duplicates(['school_id_giga', 'month'])
_cohort = sch_last['first'].dt.to_period('M').rename('cohort')
_act = _act.join(_cohort, on='school_id_giga')
_act['age'] = (_act['month'] - _act['cohort']).apply(lambda x: x.n)
_size = _cohort.value_counts().sort_index()
_ret = (_act.groupby(['cohort', 'age'])['school_id_giga'].nunique()
        .unstack(fill_value=0).div(_size, axis=0) * 100)
_ret = _ret.loc[_size[_size >= 5].index]            # cohorts of >=5 schools
_ret = _ret[[c for c in _ret.columns if c <= 18]]

# Mask months a cohort could not have reached yet: a June-2025 cohort has no
# month 15 in an August-2026 dataset — that is unobserved, not 0% retention.
_ref_period = REF_DATE.to_period('M')
for _coh in _ret.index:
    _max_age = (_ref_period - _coh).n
    _ret.loc[_coh, [c for c in _ret.columns if c > _max_age]] = np.nan

from matplotlib import colormaps as _cmaps
_cmap = _cmaps['RdYlGn'].copy(); _cmap.set_bad(GIGA_GREY[100])   # unobserved = light grey
fig, ax = plt.subplots(figsize=(12, max(3, 0.32 * len(_ret))))
im = ax.imshow(np.ma.masked_invalid(_ret.values), aspect='auto', cmap=_cmap, vmin=0, vmax=100)
ax.set_xticks(range(_ret.shape[1])); ax.set_xticklabels(_ret.columns, fontsize=8)
ax.set_yticks(range(len(_ret)))
ax.set_yticklabels([f'{i} (n={_size[i]})' for i in _ret.index], fontsize=8)
ax.set_xlabel('months since first measurement'); ax.set_ylabel('install cohort')
ax.set_title(f'Cohort retention — % of the cohort still measuring — {COUNTRY_NAME}'
             '  (grey = not yet reachable)')
plt.colorbar(im, ax=ax, fraction=0.02, pad=0.01, label='% still measuring')
plt.tight_layout(); plt.show()
if 3 in _ret.columns and 12 in _ret.columns:
    print(f"Median cohort retention: {_ret[3].median():.0f}% at 3 months, "
          f"{_ret[6].median():.0f}% at 6 months, {_ret[12].median():.0f}% at 12 months.")

---
## Q3 — What kind of school drops?

Each school is classified **dropped** (silent >30 days) or **active**, then compared
across the attributes we hold. Rates carry Wilson 95% intervals — with small
subgroups the interval is the point, not the percentage.

In [ ]:
# =============================================================================
# Q3 — WHO DROPS: RATES BY SUBGROUP (Wilson 95% CI)
# =============================================================================
prof = sch_last.reset_index()[['school_id_giga', 'first', 'last', 'n', 'days_silent']]
prof['dropoff_class'] = prof['school_id_giga'].map(CLASS)
prof['dropped'] = prof['dropoff_class'].isin(DROPPED_CLASSES)   # season-aware
prof['tenure_days'] = (prof['last'] - prof['first']).dt.days
prof['tests_per_active_month'] = prof['n'] / ((prof['tenure_days'] / 30.44).clip(lower=1))

_mcols = [c for c in ['education_level', 'admin1', 'admin2', 'school_area_type',
                      'connectivity_type_govt', 'connectivity'] if c in master.columns]
prof = prof.merge(master[['school_id_giga'] + _mcols].drop_duplicates('school_id_giga'),
                  on='school_id_giga', how='left')
_rcols = [c for c in ['num_devices_registered', 'num_devices_measured',
                      'max_app_version_gigameter', 'install_status'] if c in r.columns]
prof = prof.merge(r[['school_id_giga'] + _rcols].drop_duplicates('school_id_giga'),
                  on='school_id_giga', how='left')
_isp_col = 'isp_mapped' if 'isp_mapped' in m.columns else 'isp_name'
prof = prof.merge(m.dropna(subset=[_isp_col]).groupby('school_id_giga')[_isp_col]
                  .agg(lambda s: s.mode().iloc[0]).rename('primary_isp'),
                  on='school_id_giga', how='left')
if 'num_devices_registered' in prof.columns:
    prof['devices_band'] = pd.cut(pd.to_numeric(prof['num_devices_registered'], errors='coerce'),
                                  [-0.1, 1, 2, 3, 100], labels=['1 device', '2', '3', '4+'])
prof['app_major'] = (prof['max_app_version_gigameter'].astype(str).str.extract(r'^(\d+)')[0]
                     if 'max_app_version_gigameter' in prof.columns else np.nan)

_base = prof['dropped'].mean()
print(f"Overall drop-off rate (silent >{DROP_DAYS}d): {fmt_pct_ci(int(prof['dropped'].sum()), len(prof))} "
      f"of {len(prof):,} schools that ever measured")

def _rate_by(col, min_n=8):
    if col not in prof.columns or prof[col].notna().sum() == 0:
        print(f"({col}: not populated — skipped)"); return None
    g = prof.dropna(subset=[col]).groupby(col, observed=True)['dropped'].agg(['sum', 'size'])
    g = g[g['size'] >= min_n]
    if g.empty:
        print(f"({col}: no group with >= {min_n} schools)"); return None
    out = pd.DataFrame({'schools': g['size'].astype(int),
                        'dropped': g['sum'].astype(int),
                        'drop-off rate': [fmt_pct_ci(int(k), int(n)) for k, n in zip(g['sum'], g['size'])],
                        '_rate': (g['sum'] / g['size'] * 100).round(1)}).sort_values('_rate', ascending=False)
    print(f"\nDrop-off by {col} (overall {100*_base:.0f}%):")
    display(out.drop(columns='_rate'))
    return out

for _c in ['education_level', 'primary_isp', 'devices_band', 'app_major',
           'admin1', 'connectivity_type_govt', 'install_status']:
    _rate_by(_c)

---
## Q4 — Why? Signals behind the drop-off

Two tests. First, **numeric correlates**: do dropped schools differ from active ones
on tenure, testing intensity, and connectivity quality — compared with a rank test
and a bootstrap CI, so a difference has to survive the noise. Second, **did quality
degrade before the end**: each dropped school's last 30 measuring days against its
own earlier baseline, which controls for the school being slow in general.

In [ ]:
# =============================================================================
# Q4a — NUMERIC CORRELATES: DROPPED vs ACTIVE (rank test + bootstrap CI)
# =============================================================================
_perf = (m.groupby('school_id_giga')
         .agg(dl_median=('download_speed', 'median'), lat_median=('latency', 'median'),
              loss_median=('loss_rate', 'median')))
prof = prof.merge(_perf, on='school_id_giga', how='left')
prof['status'] = np.where(prof['dropped'], 'Dropped', 'Active')

_numeric = [('tenure_days', 'tenure (days)', 'up'),
            ('tests_per_active_month', 'tests / active month', 'up'),
            ('n', 'total measurements', 'up'),
            ('dl_median', 'median download (Mbps)', 'up'),
            ('lat_median', 'median latency (ms)', 'down'),
            ('num_devices_registered', 'devices registered', 'up')]
print(f"Dropped (n={int(prof['dropped'].sum()):,}) vs Active (n={int((~prof['dropped']).sum()):,}) — "
      f"per-school values, Mann-Whitney + bootstrap 95% CI on the median shift:")
for _col, _lbl, _better in _numeric:
    if _col not in prof.columns or prof[_col].notna().sum() < 20:
        continue
    # one row per school here, so a unit needs only 1 observation per cell
    _res = two_group_shift_test(prof.dropna(subset=[_col]), unit_col='school_id_giga',
                                group_col='status', value_col=_col,
                                group_a='Dropped', group_b='Active', min_per_cell=1)
    print(format_shift_result(_res, label=f'  {_lbl}', better=_better))

In [ ]:
# =============================================================================
# Q4b — DID QUALITY DEGRADE BEFORE THE END? (each school its own control)
# =============================================================================
# For every dropped school: its last 30 measuring days vs everything before,
# paired per school. A real degradation shows as a negative shift on download.
_dropped_ids = set(prof.loc[prof['dropped'], 'school_id_giga'])
_md = m[m['school_id_giga'].isin(_dropped_ids)].merge(
    sch_last[['last']], left_on='school_id_giga', right_index=True, how='left')
_md['phase'] = np.where(_md['date'] > _md['last'] - pd.Timedelta(days=30), 'Final 30 days', 'Earlier')

_n_pairs = (_md.groupby('school_id_giga')['phase'].nunique() == 2).sum()
print(f"Dropped schools with measurements in both phases: {_n_pairs:,}")
for _col, _lbl, _better in [('download_speed', 'download (Mbps)', 'up'),
                            ('upload_speed', 'upload (Mbps)', 'up'),
                            ('latency', 'latency (ms)', 'down')]:
    if _col not in _md.columns or _md[_col].notna().sum() == 0:
        continue
    _res = paired_shift_test(_md, unit_col='school_id_giga', group_col='phase', value_col=_col,
                             group_a='Final 30 days', group_b='Earlier', min_per_cell=3)
    print(format_shift_result(_res, label=f'  {_lbl}', better=_better))

# Measurement CADENCE in the final weeks — did testing thin out before stopping?
# Only whole weeks INSIDE the window — clipping everything older into one bucket
# would pile a school's entire history into it (and read as a spike, not a week).
WEEKS_BACK = 12
_md['weeks_to_end'] = (_md['last'] - _md['date']).dt.days // 7
_md_w = _md[_md['weeks_to_end'] < WEEKS_BACK]
_cad = (_md_w.groupby(['school_id_giga', 'weeks_to_end'])['date'].nunique()
        .groupby('weeks_to_end').median().sort_index(ascending=False))
_n_sch = _md_w.groupby('weeks_to_end')['school_id_giga'].nunique().sort_index(ascending=False)
fig, ax = plt.subplots(figsize=(10, 3.6))
ax.plot(_cad.index, _cad.values, marker='o', color=GIGA_PRIMARY[600])
ax.invert_xaxis()
ax.set_ylim(0, 7.5)
ax.set_xlabel('weeks before the last measurement (0 = final week)')
ax.set_ylabel('median measuring days / week')
ax.set_title(f'Did measurement thin out before the end? — {COUNTRY_NAME}')
plt.tight_layout(); plt.show()
print(f"schools contributing per week: {_n_sch.min():,}-{_n_sch.max():,}")
print("A flat line means schools stop abruptly (device/person event); a decline means")
print("engagement faded first — different remedies.")

---
## Q5 — What predicts churn?

Silence spells labelled by their observed outcome, compared across signals
available at the moment the pause began.


In [ ]:
# =============================================================================
# Q5 — WHAT PREDICTS CHURN? (labelled outcomes, not assumptions)
# =============================================================================
# A single long gap is not proof of churn — most end in a return. To find what
# IS predictive, every silence spell that began at least HORIZON days before the
# reference date is labelled by its observed outcome (returned or not), then
# compared across signals available AT THE START of the pause.
HORIZON = 180

_sdd = m_all.drop_duplicates(['school_id_giga', 'date']).sort_values(['school_id_giga', 'date'])
_rows = []
for _s, _g in _sdd.groupby('school_id_giga'):
    _d = _g['date'].tolist()
    for _i in range(1, len(_d)):
        if (_d[_i] - _d[_i - 1]).days > 30:
            _rows.append((_s, _d[_i - 1], (_d[_i] - _d[_i - 1]).days, True))
    if (REF_DATE - _d[-1]).days > 30:
        _rows.append((_s, _d[-1], (REF_DATE - _d[-1]).days, False))
sp = pd.DataFrame(_rows, columns=['school_id_giga', 'pause_start', 'length', 'returned'])
sp = sp[sp['pause_start'] <= REF_DATE - pd.Timedelta(days=HORIZON)].copy()
sp['returned_h'] = sp['returned'] & (sp['length'] <= HORIZON)

sp['tenure_days'] = (sp['pause_start'] - sp['school_id_giga'].map(sch_last['first'])).dt.days
sp['started_low_season'] = sp['pause_start'].dt.month.isin(LOW_SEASON_MONTHS)
sp['prior_returns'] = [int(((sp['school_id_giga'] == s) & (sp['pause_start'] < t) & sp['returned']).sum())
                       for s, t in zip(sp['school_id_giga'], sp['pause_start'])]
sp['days_measured_per_month'] = [
    len(_sdd[(_sdd['school_id_giga'] == s) & (_sdd['date'] <= t) &
             (_sdd['date'] > t - pd.Timedelta(days=90))]) / 3
    for s, t in zip(sp['school_id_giga'], sp['pause_start'])]

print(f"{len(sp):,} silence spells with a full {HORIZON}-day outcome window · "
      f"{100 * sp['returned_h'].mean():.0f}% returned within {HORIZON} days\n")

def _return_rate(col, bins=None, labels=None, min_n=15):
    _x = pd.cut(sp[col], bins, labels=labels) if bins is not None else sp[col]
    _t = sp.groupby(_x, observed=True)['returned_h'].agg(['size', 'sum'])
    _t = _t[_t['size'] >= min_n]
    if _t.empty:
        return
    _t['return rate'] = [fmt_pct_ci(int(k), int(n)) for k, n in zip(_t['sum'], _t['size'])]
    print(f"{col}:")
    display(_t[['size', 'return rate']].rename(columns={'size': 'spells'}))

_return_rate('started_low_season')
# NOTE: spell LENGTH is deliberately not shown here — it is the outcome, not a
# predictor (a 31-60d spell trivially 'returned within 180d', a 183d+ one cannot).
# The legitimate version is the conditional survival table in the calibration cell.
_return_rate('tenure_days', [-1, 90, 365, 730, 10 ** 5], ['<3 mo', '3-12 mo', '1-2 yr', '2 yr+'])
_return_rate('prior_returns', [-1, 0, 1, 2, 99], ['0', '1', '2', '3+'])
_return_rate('days_measured_per_month', [-1, 5, 15, 25, 99], ['<5', '5-15', '15-25', '25+'])

print("Read it as: signals whose intervals do not overlap are the ones worth acting on. The strong")
print("ones are usually WHEN the pause began (low season = probably a break) and HOW LONG it has run;")
print("engagement intensity before the pause tends to be weak or even inverted.")

In [ ]:
# =============================================================================
# EXPORT — per-school drop-off profile
# =============================================================================
_out = Path(CACHE_DIR) / 'superset'
_out.mkdir(exist_ok=True)
_cols = [c for c in ['school_id_giga', 'dropoff_class', 'first', 'last', 'days_silent', 'dropped', 'tenure_days',
                     'n', 'tests_per_active_month', 'education_level', 'admin1', 'admin2',
                     'primary_isp', 'num_devices_registered', 'max_app_version_gigameter',
                     'install_status', 'dl_median', 'lat_median'] if c in prof.columns]
prof[_cols].to_csv(_out / f'{COUNTRY_ISO3.lower()}_dropoff_profile.csv', index=False)
print(f"exported {_out / f'{COUNTRY_ISO3.lower()}_dropoff_profile.csv'} ({len(prof):,} schools)")